# Project setup

In [7]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "shared" / "config.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/sanjeet/ai_workspace/KWS model/sih-2026


# Load configuration

In [8]:
from shared.config import KEYWORD, MODELS_DIR
from shared.feature_spec import SPEC, FEATURE_SHAPE
from training.model import N_CLASSES

print(f"Keyword       : {KEYWORD}")
print(f"Feature shape : {FEATURE_SHAPE}")
print(f"Classes       : {N_CLASSES}")

Keyword       : marvin
Feature shape : (49, 20, 1)
Classes       : 3


# Load current validation set

In [9]:
from training.train import collect

train_data, val_data, speakers, gsc_split = collect(
    KEYWORD,
    holdout_speaker=None,
    noise_cap=400,
)

x_val, y_val, _ = val_data.arrays()

print(f"Validation samples : {len(x_val)}")
print(f"Feature shape      : {x_val.shape[1:]}")
print(f"Using GSC test split: {gsc_split}")

  loaded 2000 clips...
  loaded 3000 clips...
  loaded 4000 clips...
  loaded 6000 clips...
  loaded 7000 clips...
  loaded 8000 clips...
  loaded 9000 clips...
  loaded 10000 clips...
  loaded 11000 clips...
  loaded 12000 clips...
  loaded 13000 clips...
  loaded 14000 clips...
  loaded 15000 clips...
  loaded 16000 clips...
  loaded 17000 clips...
  loaded 18000 clips...
  loaded 19000 clips...
  loaded 20000 clips...
  loaded 21000 clips...
  loaded 22000 clips...
  loaded 23000 clips...
Validation samples : 11164
Feature shape      : (49, 20, 1)
Using GSC test split: True


# Load model

In [ ]:
import tensorflow as tf

model_path = ""

model = tf.keras.models.load_model(
    model_path,
    compile=False,
)

model.summary()

/home/sanjeet/ai_workspace/KWS model/sih-2026/models/marvin_pruned_finetuned_masked.keras
True


W0000 00:00:1790065022.208356   93801 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


TypeError: <class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': None}.

Exception encountered: Could not locate class 'PersistentMaskedActivation'. Make sure custom classes and functions are decorated with `@keras.saving.register_keras_serializable()`. If they are already decorated, make sure they are all imported so that the decorator is run before trying to load them. Full object config: {'module': None, 'class_name': 'PersistentMaskedActivation', 'config': {'name': 'stem_relu', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_

In [12]:
import tensorflow as tf
from pathlib import Path

model_path = PROJECT_ROOT / "models" / "marvin.keras"
models_dir = PROJECT_ROOT / "models"

# ---------- FP16 ----------
fp16_converter = tf.lite.TFLiteConverter.from_keras_model(
    tf.keras.models.load_model(model_path, compile=False)
)

fp16_converter.optimizations = [tf.lite.Optimize.DEFAULT]
fp16_converter.target_spec.supported_types = [tf.float16]

fp16_model = fp16_converter.convert()

fp16_path = models_dir / "marvin.fp16.tflite"
fp16_path.write_bytes(fp16_model)


# ---------- INT8 ----------
int8_converter = tf.lite.TFLiteConverter.from_keras_model(
    tf.keras.models.load_model(model_path, compile=False)
)

int8_converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Representative dataset for activation calibration
def representative_dataset():
    for i in range(min(len(x_val), 500)):
        yield [x_val[i:i+1].astype("float32")]

int8_converter.representative_dataset = representative_dataset

int8_converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

int8_converter.inference_input_type = tf.int8
int8_converter.inference_output_type = tf.int8

int8_model = int8_converter.convert()

int8_path = models_dir / "marvin.int8.tflite"
int8_path.write_bytes(int8_model)


print(f"FP16: {fp16_path} — {len(fp16_model):,} bytes")
print(f"INT8: {int8_path} — {len(int8_model):,} bytes")

INFO:tensorflow:Assets written to: /tmp/tmpi8cqmfhp/assets


INFO:tensorflow:Assets written to: /tmp/tmpi8cqmfhp/assets


Saved artifact at '/tmp/tmpi8cqmfhp'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 49, 20, 1), dtype=tf.float32, name='mfcc')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  127996383766352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996383765200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996383763856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996383764240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996383764048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996383765968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996383765776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996383764432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996383765392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996383762896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996383763472: TensorS

W0000 00:00:1789896952.449011   42682 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1789896952.449455   42682 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1789896952.456603   42682 reader.cc:83] Reading SavedModel from: /tmp/tmpi8cqmfhp
I0000 00:00:1789896952.459483   42682 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1789896952.459508   42682 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpi8cqmfhp
I0000 00:00:1789896952.491428   42682 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1789896952.639529   42682 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpi8cqmfhp
I0000 00:00:1789896952.672883   42682 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 216301 microseconds.


INFO:tensorflow:Assets written to: /tmp/tmpc_iv943r/assets


INFO:tensorflow:Assets written to: /tmp/tmpc_iv943r/assets


Saved artifact at '/tmp/tmpc_iv943r'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 49, 20, 1), dtype=tf.float32, name='mfcc')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  127996338714192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996338714000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996338715728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996338716112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996338715920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996338713808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996338716496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996338716304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996338713616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996338714768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996338715344: TensorS

/home/sanjeet/ai_workspace/KWS model/sih-2026/.venv/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1789896954.973459   42682 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1789896954.973527   42682 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1789896954.973722   42682 reader.cc:83] Reading SavedModel from: /tmp/tmpc_iv943r
I0000 00:00:1789896954.976546   42682 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1789896954.976584   42682 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpc_iv943r
I0000 00:00:1789896954.999738   42682 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1789896955.112433   42682 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpc_iv943r
I0000 00:00:1789896955.142176   42682 loader.cc:471] 

FP16: /home/sanjeet/ai_workspace/KWS model/sih-2026/models/marvin.fp16.tflite — 59,072 bytes
INT8: /home/sanjeet/ai_workspace/KWS model/sih-2026/models/marvin.int8.tflite — 48,888 bytes


fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
W0000 00:00:1789896956.150304   42682 flatbuffer_export.cc:3851] Skipping runtime version metadata in the model. This will be generated by the exporter.
